# Server Load Prediction


### Dummy Dataset Creation

In [ ]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Number of rows
n_rows = 2000

# Generate synthetic dataset for server load forecasting
timestamps = pd.date_range(start="2024-01-01", periods=n_rows, freq="H")

# Features
cpu_usage = np.random.uniform(10, 95, size=n_rows)  # CPU usage percentage
memory_usage = np.random.uniform(20, 95, size=n_rows)  # Memory usage percentage
disk_io = np.random.uniform(50, 500, size=n_rows)  # Disk I/O in MB/s
network_in = np.random.uniform(10, 1000, size=n_rows)  # Network inbound in MB/s
network_out = np.random.uniform(10, 1000, size=n_rows)  # Network outbound in MB/s
active_users = np.random.randint(50, 5000, size=n_rows)  # Active users count

# Target variable (server load) influenced by CPU, Memory, and Active Users
server_load = (
    0.4 * cpu_usage +
    0.3 * memory_usage +
    0.2 * (active_users / 100) +
    np.random.normal(0, 5, size=n_rows)  # Noise
)

# Create DataFrame
df = pd.DataFrame({
    "Timestamp": timestamps,
    "CPU_Usage(%)": cpu_usage,
    "Memory_Usage(%)": memory_usage,
    "Disk_IO(MBps)": disk_io,
    "Network_In(MBps)": network_in,
    "Network_Out(MBps)": network_out,
    "Active_Users": active_users,
    "Server_Load(%)": server_load
})

# Save to CSV
file_path = "server_load_dataset.csv"
df.to_csv(file_path, index=False)

file_path

### Import Needed Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import PolynomialFeatures

import joblib

from sklearn import metrics

from warnings import filterwarnings
filterwarnings('ignore')

### Data Collection and Processing



In [3]:
#Read the data
df = pd.read_csv(r'server_load_dataset.csv')

In [4]:
#See first 10 rows
df.head(10)

,Timestamp,CPU_Usage(%),Memory_Usage(%),Disk_IO(MBps),Network_In(MBps),Network_Out(MBps),Active_Users,Server_Load(%)
0,2024-01-01 00:00:00,41.835910,39.627926,307.398145,651.774385,723.065245,3304,38.343877
1,2024-01-01 01:00:00,90.810716,38.523410,412.444548,180.662498,690.410174,913,52.968450
2,2024-01-01 02:00:00,72.219485,87.969094,392.072418,873.670618,104.796657,4003,68.125122
3,2024-01-01 03:00:00,60.885971,38.715965,119.254957,616.985077,923.346681,3158,32.856888
4,2024-01-01 04:00:00,23.261584,40.396229,117.162261,165.631845,572.787480,3105,30.178733
5,2024-01-01 05:00:00,23.259534,76.954870,170.678465,962.714677,370.088266,4044,44.984623
6,2024-01-01 06:00:00,14.937107,53.730488,212.483627,523.181809,758.973198,2732,25.500696
7,2024-01-01 07:00:00,83.624972,78.253292,233.805012,82.169467,264.791804,4740,63.666869
8,2024-01-01 08:00:00,61.094776,24.902462,355.863748,630.564575,696.574739,663,29.727761
9,2024-01-01 09:00:00,70.186169,56.567840,75.506194,260.666918,49.314095,1922,43.728791


In [5]:
#print number of rows and columns separately

print("Number of Rows:",df.shape[0])
print("Number of Features:",df.shape[1])


Number of Rows: 2000
Number of Features: 8


In [6]:
#see dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Timestamp          2000 non-null   object 
 1   CPU_Usage(%)       2000 non-null   float64
 2   Memory_Usage(%)    2000 non-null   float64
 3   Disk_IO(MBps)      2000 non-null   float64
 4   Network_In(MBps)   2000 non-null   float64
 5   Network_Out(MBps)  2000 non-null   float64
 6   Active_Users       2000 non-null   int64  
 7   Server_Load(%)     2000 non-null   float64
dtypes: float64(6), int64(1), object(1)
memory usage: 125.1+ KB


In [7]:
# define numerical & categorical columns
numeric_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

# print columns
print('We have {} numerical features : {}'.format(len(numeric_features), numeric_features))
print('\nWe have {} categorical features : {}'.format(len(categorical_features), categorical_features))

We have 7 numerical features : ['CPU_Usage(%)', 'Memory_Usage(%)', 'Disk_IO(MBps)', 'Network_In(MBps)', 'Network_Out(MBps)', 'Active_Users', 'Server_Load(%)']

We have 1 categorical features : ['Timestamp']


In [8]:
#Check for missing values
df.isnull().sum()

Timestamp            0
CPU_Usage(%)         0
Memory_Usage(%)      0
Disk_IO(MBps)        0
Network_In(MBps)     0
Network_Out(MBps)    0
Active_Users         0
Server_Load(%)       0
dtype: int64

### Data Analysis

In [9]:
#See descriptive statistics of numerical columns
df.describe()

,CPU_Usage(%),Memory_Usage(%),Disk_IO(MBps),Network_In(MBps),Network_Out(MBps),Active_Users,Server_Load(%)
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.0000,2000.000000
mean,52.384139,57.229279,273.328208,492.961331,496.729380,2507.0090,43.004164
std,24.837918,21.643492,129.573473,284.441185,279.348595,1425.1997,13.111860
min,10.273552,20.000873,50.013823,10.238595,10.052299,54.0000,10.013085
25%,30.233950,38.809007,161.120809,246.936848,260.729702,1288.7500,33.291470
50%,53.124858,56.952075,271.179836,489.356831,491.836796,2502.0000,42.751525
75%,73.807701,76.151788,382.933300,739.762279,732.386880,3762.5000,52.316858
max,94.976002,94.966828,499.707635,999.466074,999.510138,4996.0000,82.515637


### Seperate Features from Label

In [11]:
#sepertate features and target

X = df.iloc[:, 1:-1]

y = df.iloc[:, -1]

In [12]:
X

,CPU_Usage(%),Memory_Usage(%),Disk_IO(MBps),Network_In(MBps),Network_Out(MBps),Active_Users
0,41.835910,39.627926,307.398145,651.774385,723.065245,3304
1,90.810716,38.523410,412.444548,180.662498,690.410174,913
2,72.219485,87.969094,392.072418,873.670618,104.796657,4003
3,60.885971,38.715965,119.254957,616.985077,923.346681,3158
4,23.261584,40.396229,117.162261,165.631845,572.787480,3105
...,...,...,...,...,...,...
1995,65.841188,53.158028,357.995912,280.428024,859.079428,4368
1996,91.312243,45.080088,276.449187,217.424032,898.533747,2690
1997,15.861431,49.592924,394.316982,460.775462,947.240836,1306
1998,14.849651,59.745544,268.380785,909.353172,403.513112,877


In [13]:
y

0       38.343877
1       52.968450
2       68.125122
3       32.856888
4       30.178733
          ...    
1995    41.330933
1996    49.321781
1997    22.389422
1998    19.942212
1999    26.139338
Name: Server_Load(%), Length: 2000, dtype: float64

### Split data into train and test sets

In [15]:
#splittting data into training and testing data
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
#print shape of features and training and testing data of features
print("Shape of Features:", X.shape)
print("Shape of Features_train:", X_train.shape)
print("Shape of Features_test:", X_test.shape)

Shape of Features: (2000, 6)
Shape of Features_train: (1600, 6)
Shape of Features_test: (400, 6)


In [17]:
#print shape of Target and training and testing data of Target
print("Shape of Target:", y.shape)
print("Shape of Target_train:", y_train.shape)
print("Shape of Target_test:", y_test.shape)

Shape of Target: (2000,)
Shape of Target_train: (1600,)
Shape of Target_test: (400,)


### Building Model


In [18]:
models_r2_score = {
    "Multiple Linear Regression": None,
    "Decision Tree Regressor": None,
    "Random Forest Regressor": None,
    "XGBoost Regressor": None,
    "Support Vector Regressor": None
}

#### Multiple Linear Regression

In [19]:
# ---------- 1. Multiple Linear Regression ----------
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
Target_pred = lr_model.predict(X_test)

#calculate R-Squared
r2_score = metrics.r2_score(y_test,Target_pred)
print("R-Squared:",r2_score)
models_r2_score["Multiple Linear Regression"] = [lr_model, round(r2_score,4)]

#calculate Mean Absolute Error
mae = metrics.mean_absolute_error(y_test,Target_pred)
print("Mean Absolute Error:",mae)

#calculate Mean Squared Error
mse = metrics.mean_squared_error(y_test,Target_pred)
print("Mean Squared Error:",mse)

R-Squared: 0.8605557797395845
Mean Absolute Error: 3.9562257495037922
Mean Squared Error: 23.86997882783412


#### Decision Trees Regressor

In [20]:
# ---------- 3. Random Forest Regressor ----------
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
pred_dt = dt_model.predict(X_test)

#calculate R-Squared
r2_score = metrics.r2_score(y_test,pred_dt)
print("R-Squared:",r2_score)
models_r2_score["Decision Tree Regressor"] = [dt_model, round(r2_score,4)]

#calculate Mean Absolute Error
mae = metrics.mean_absolute_error(y_test,pred_dt)
print("Mean Absolute Error:",mae)

#calculate Mean Squared Error
mse = metrics.mean_squared_error(y_test,pred_dt)
print("Mean Squared Error:",mse)

R-Squared: 0.6600958293191044
Mean Absolute Error: 6.202580591234806
Mean Squared Error: 58.18459411579286


#### Building  XGBRegressor Model

In [21]:
#build model with XGBRegressor
XGBRModel = XGBRegressor()
XGBRModel.fit(X_train,y_train)
Target_pred = XGBRModel.predict(X_test)

#calculate R-Squared
r2_score = metrics.r2_score(y_test,Target_pred)
print("R-Squared:",r2_score)
models_r2_score["XGBoost Regressor"] = [XGBRModel, round(r2_score,4)]

#calculate Mean Absolute Error
mae = metrics.mean_absolute_error(y_test,Target_pred)
print("Mean Absolute Error:",mae)

#calculate Mean Squared Error
mse = metrics.mean_squared_error(y_test,Target_pred)
print("Mean Squared Error:",mse)


R-Squared: 0.8076083807149983
Mean Absolute Error: 4.6218734816865545
Mean Squared Error: 32.93348315483659


#### Building  SVR Model

In [22]:
#build model with SVR
svr_model = SVR()
svr_model.fit(X_train,y_train)
Target_pred = svr_model.predict(X_test)

#calculate R-Squared
r2_score = metrics.r2_score(y_test,Target_pred)
print("R-Squared:",r2_score)
models_r2_score["Support Vector Regressor"] =  [svr_model, round(r2_score,4)]

#calculate Mean Absolute Error
mae = metrics.mean_absolute_error(y_test,Target_pred)
print("Mean Absolute Error:",mae)

#calculate Mean Squared Error
mse = metrics.mean_squared_error(y_test,Target_pred)
print("Mean Squared Error:",mse)

R-Squared: 0.04357272341946106
Mean Absolute Error: 10.47276699231179
Mean Squared Error: 163.72065331718395


#### Building RandomForestRegressor Model

In [23]:
#build model with RandomForestRegressor
rf_model = RandomForestRegressor()
rf_model.fit(X_train,y_train)
Target_pred = rf_model.predict(X_test)

#calculate R-Squared
r2_score = metrics.r2_score(y_test,Target_pred)
print("R-Squared:",r2_score)
models_r2_score["Random Forest Regressor"] = [rf_model, round(r2_score,4)]


#calculate Mean Absolute Error
mae = metrics.mean_absolute_error(y_test,Target_pred)
print("Mean Absolute Error:",mae)

#calculate Mean Squared Error
mse = metrics.mean_squared_error(y_test,Target_pred)
print("Mean Squared Error:",mse)


R-Squared: 0.8363209846253378
Mean Absolute Error: 4.300102844303947
Mean Squared Error: 28.018476665848738


#### Finding and saving best model

In [24]:
models_r2_score

{'Multiple Linear Regression': [LinearRegression(), 0.8606],
 'Decision Tree Regressor': [DecisionTreeRegressor(random_state=42), 0.6601],
 'Random Forest Regressor': [RandomForestRegressor(), 0.8363],
 'XGBoost Regressor': [XGBRegressor(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=None, num_parallel_tree=None, ...),
  0.

In [25]:
import joblib

# Pick the best model based on R² score (index 1 of the list)
best_model_name = max(models_r2_score, key=lambda name: models_r2_score[name][1])

# Unpack the model and R² score
best_model, best_r2 = models_r2_score[best_model_name]

# Save only the trained model
joblib.dump(best_model, "best_model.pkl")

print(f"Best Model: {best_model_name} (R² = {best_r2:.4f}) saved as best_model.pkl")



Best Model: Multiple Linear Regression (R² = 0.8606) saved as best_model.pkl


### Make a predictive System

In [26]:
# Preprocess the first 5 rows of your original data
X_sample = X.head(5)
y_sample = y.head(5)  # Assuming your target variable is stored in `y`

# Predict
predictions = best_model.predict(X_sample)

# Create a comparison DataFrame
comparison_df = pd.DataFrame({
    "Actual": y_sample.values,
    "Predicted": predictions
})

print(comparison_df)

      Actual  Predicted
0  38.343877  35.168287
1  52.968450  49.443221
2  68.125122  62.840103
3  32.856888  42.219976
4  30.178733  27.454532


In [ ]:
import pandas as pd
import joblib

# Load
best_model = joblib.load("best_model.pkl")
# New input
new_data = [[41.835910, 39.627926, 307.398145, 651.774385, 723.065245, 3304]]
# Predict
prediction = best_model.predict(new_data)
print("Predicted value:", prediction[0])


Predicted value: 35.168287156267134
